In [ ]:
import urllib.request
import zipfile
import zipfile
import os
import pandas as pd

# Path to zipped file in Downloads
zip_path = r"C:\Users\HP\Downloads\ml-latest-small.zip" # Update if your zip has a slightly different name
extract_path = r"C:\Users\HP\Downloads\ml-latest-small_extracted"

# Extract the zip file using Python
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")

# load the CSV files from the extracted folder
ratings_df = pd.read_csv(os.path.join(extract_path, "ml-latest-small", "ratings.csv"))
movies_df = pd.read_csv(os.path.join(extract_path, "ml-latest-small", "movies.csv"))

print("Successfully loaded!")
print("Ratings shape:", ratings_df.shape)
print("Movies shape:", movies_df.shape)

Successfully loaded!
Ratings shape: (100836, 4)
Movies shape: (9742, 3)


In [14]:
# Step 2: Basic Data Inspection & Exploration
print("--- Ratings Info ---")
print(ratings_df.info())
print("\n--- Missing Values in Ratings ---")
print(ratings_df.isnull().sum())

print("\n--- Movies Info ---")
print(movies_df.info())
print (movies_df.isnull().sum())

# Check unique users and movies and calculate matrix sparsity
n_users = ratings_df['userId'].nunique()
n_movies = ratings_df['movieId'].nunique()

print(f"\nUnique Users: {n_users}")
print(f"Unique Movies: {n_movies}")
print(f"Total Possible Ratings: {n_users * n_movies:,}")
print(f"Actual Ratings: {len(ratings_df):,}")

sparsity = (1 - (len(ratings_df) / (n_users * n_movies))) * 100
print(f"Matrix Sparsity: {sparsity:.2f}%")

--- Ratings Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
None

--- Missing Values in Ratings ---
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

--- Movies Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB
None
movieId    0
title      0
genres     0
dtype: int64

Unique Users: 610
Unique Movies: 9724


In [ ]:
# Step 3: Filter out inactive users and unpopular movies
# - Minimum ratings required for a user to be included: e.g., 20
# - Minimum ratings required for a movie to be included: e.g., 10

min_user_ratings = 20
min_movie_ratings = 10

# Filter users
counts_user = ratings_df['userId'].value_counts()
active_users = counts_user[counts_user >= min_user_ratings].index
filtered_ratings = ratings_df[ratings_df['userId'].isin(active_users)]

# Filter movies
counts_movie = filtered_ratings['movieId'].value_counts()
popular_movies = counts_movie[counts_movie >= min_movie_ratings].index
filtered_ratings = filtered_ratings[filtered_ratings['movieId'].isin(popular_movies)]

print(f"Original ratings shape: {ratings_df.shape}")
print(f"Filtered ratings shape: {filtered_ratings.shape}")

# Recalculate unique users, movies, and sparsity after filtering
n_users_filtered = filtered_ratings['userId'].nunique()
n_movies_filtered = filtered_ratings['movieId'].nunique()
new_sparsity = (1 - (len(filtered_ratings) / (n_users_filtered * n_movies_filtered))) * 100

print(f"Filtered Unique Users: {n_users_filtered}")
print(f"Filtered Unique Movies: {n_movies_filtered}")
print(f"New Matrix Sparsity: {new_sparsity:.2f}%")

Original ratings shape: (100836, 4)
Filtered ratings shape: (81116, 4)
Filtered Unique Users: 610
Filtered Unique Movies: 2269
New Matrix Sparsity: 94.14%


In [6]:
# Step 4: Create the User-Item Pivot Matrix
# Rows = userId, Columns = movieId, Values = rating
user_movie_matrix = filtered_ratings.pivot(
    index='userId', 
    columns='movieId', 
    values='rating'
).fillna(0) # Fill unrated slots with 0 for matrix operations

print("User-Movie Matrix Shape:", user_movie_matrix.shape)
print(user_movie_matrix.head())

User-Movie Matrix Shape: (610, 2269)
movieId  1       2       3       5       6       7       9       10      \
userId                                                                    
1           4.0     0.0     4.0     0.0     4.0     0.0     0.0     0.0   
2           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
5           4.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   

movieId  11      12      ...  166461  166528  166643  168250  168252  174055  \
userId                   ...                                                   
1           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
2           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0  ...     0.0 

In [7]:
# Step 5: Train the KNN Collaborative Filtering Model
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

# Convert our pandas dataframe into a Scipy sparse matrix for efficiency
user_movie_sparse = csr_matrix(user_movie_matrix.values)

# Initialize the NearestNeighbors model
# metric='cosine' measures the angle between user rating vectors (ignoring magnitude differences)
# algorithm='brute' evaluates all pairs (best for dense or smaller sparse matrices)
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20, n_jobs=-1)

# Fit the model on the sparse user-movie matrix
model_knn.fit(user_movie_sparse)

print("KNN Model successfully trained!")

KNN Model successfully trained!


In [18]:
# Step 6: Function to generate top 5 movie recommendations for a given user
import numpy as np

def get_movie_recommendations(user_id, model, user_matrix, ratings_df, movies_df, n_recommendations=5):
    # Check if user exists in our filtered matrix
    if user_id not in user_matrix.index:
        return f"User ID {user_id} not found in the filtered training matrix."
    
    # Get the index of the user in the matrix
    user_idx = user_matrix.index.get_loc(user_id)
    
    # Reshape user vector for the KNN model
    user_vector = user_matrix.iloc[user_idx, :].values.reshape(1, -1)
    
    # Find nearest neighbors (similar users)
    distances, indices = model.kneighbors(user_vector, n_neighbors=6) # 6 because the 1st neighbor is the user themselves
    
    # Extract neighbor user IDs (skipping the first one as it's the user themself)
    neighbor_indices = indices.flatten()[1:]
    neighbor_user_ids = user_matrix.index[neighbor_indices]
    
    print(f"Found {len(neighbor_user_ids)} similar users for User {user_id}.")
    
    # Gather movies rated by these neighbors
    neighbor_ratings = ratings_df[ratings_df['userId'].isin(neighbor_user_ids)]
    
    # Get movies already rated by the target user
    user_rated_movies = ratings_df[ratings_df['userId'] == user_id]['movieId'].tolist()
    
    # Filter out movies the target user has already rated/watched
    recommendable_movies = neighbor_ratings[~neighbor_ratings['movieId'].isin(user_rated_movies)]
    
    # Calculate a simple score: average rating * count of recommendations from neighbors
    movie_scores = recommendable_movies.groupby('movieId')['rating'].agg(['mean', 'count'])
    
    # Weight the score by popularity/consensus (e.g., mean rating * log(count + 1))
    movie_scores['final_score'] = movie_scores['mean'] * np.log1p(movie_scores['count'])
    
    # Sort by the highest score
    top_movie_ids = movie_scores.sort_values(by='final_score', ascending=False).head(n_recommendations).index
    
    # Merge with movies_df to get actual movie titles
    recommended_movies = movies_df[movies_df['movieId'].isin(top_movie_ids)]
    
    # Reorder according to our top_movie_ids order
    recommended_movies = recommended_movies.set_index('movieId').loc[top_movie_ids].reset_index()
    
    return recommended_movies[['movieId', 'title', 'genres']]

# Test the function on an active user from our dataset (e.g., the first user in our matrix)
sample_user = user_movie_matrix.index[0]
print(f"\n--- Generating Top 5 Recommendations for User {sample_user} ---")
recommendations = get_movie_recommendations(sample_user, model_knn, user_movie_matrix, filtered_ratings, movies_df, n_recommendations=5)
print(recommendations)


--- Generating Top 5 Recommendations for User 1 ---
Found 5 similar users for User 1.
   movieId                             title                          genres
0     1200                     Aliens (1986)  Action|Adventure|Horror|Sci-Fi
1      858             Godfather, The (1972)                     Crime|Drama
2      541               Blade Runner (1982)          Action|Sci-Fi|Thriller
3     1610  Hunt for Red October, The (1990)       Action|Adventure|Thriller
4     1036                   Die Hard (1988)           Action|Crime|Thriller


In [9]:
# Step 7: Model Evaluation using Train-Test Split and RMSE
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# Split our filtered ratings into train (80%) and test (20%) sets
train_data, test_data = train_test_split(filtered_ratings, test_size=0.2, random_state=42)

# Create a train user-movie matrix for evaluation
train_matrix = train_data.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Define a simple prediction function for evaluation: 
# Predict a user's rating for a movie based on the average rating of that movie by their nearest neighbors,
# or fall back to the movie's overall average in the training set.
def predict_rating(user_id, movie_id, model, user_matrix, train_data):
    if user_id not in user_matrix.index or movie_id not in user_matrix.columns:
        # Fall back to global movie average or overall average if unknown
        movie_avg = train_data[train_data['movieId'] == movie_id]['rating'].mean()
        return movie_avg if not np.isnan(movie_avg) else train_data['rating'].mean()
    
    user_idx = user_matrix.index.get_loc(user_id)
    user_vector = user_matrix.iloc[user_idx, :].values.reshape(1, -1)
    
    # Find neighbors
    distances, indices = model.kneighbors(user_vector, n_neighbors=6)
    neighbor_indices = indices.flatten()[1:]
    neighbor_user_ids = user_matrix.index[neighbor_indices]
    
    # Get ratings given by neighbors for this specific movie
    neighbor_ratings = train_data[(train_data['userId'].isin(neighbor_user_ids)) & (train_data['movieId'] == movie_id)]
    
    if len(neighbor_ratings) > 0:
        return neighbor_ratings['rating'].mean()
    else:
        # Fall back to movie average
        movie_avg = train_data[train_data['movieId'] == movie_id]['rating'].mean()
        return movie_avg if not np.isnan(movie_avg) else train_data['rating'].mean()

# Evaluate on a sample of the test set to save computation time
sample_test = test_data.sample(n=500, random_state=42)
predictions = []
actuals = []

for _, row in sample_test.iterrows():
    pred = predict_rating(row['userId'], row['movieId'], model_knn, train_matrix, train_data)
    predictions.append(pred)
    actuals.append(row['rating'])

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(actuals, predictions))
print(f"Evaluation Complete!")
print(f"Root Mean Squared Error (RMSE) on Test Sample: {rmse:.4f}")

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.d

Evaluation Complete!
Root Mean Squared Error (RMSE) on Test Sample: 1.0738


In [10]:
# Function to handle the cold-start problem for new or unknown users
def get_recommendations_with_cold_start(user_id, model, user_matrix, ratings_df, movies_df, n_recommendations=5):
    # Check if user exists in our trained matrix
    if user_id in user_matrix.index:
        # Use our existing collaborative filtering logic
        return get_movie_recommendations(user_id, model, user_matrix, ratings_df, movies_df, n_recommendations)
    
    else:
        print(f"User ID {user_id} is a new user (Cold Start)! Falling back to popular recommendations.")
        
        # Fallback strategy: Recommend the highest-rated movies overall 
        # (Filtered by movies that have a good number of reviews to ensure quality)
        movie_stats = ratings_df.groupby('movieId').agg({'rating': ['mean', 'count']})
        movie_stats.columns = ['mean_rating', 'review_count']
        
        # Require at least 20 ratings to avoid obscure movies with a single 5-star rating
        popular_movies = movie_stats[movie_stats['review_count'] >= 20]
        
        # Calculate a weighted score (Bayesian average or simple mean * review count weight)
        popular_movies['score'] = popular_movies['mean_rating'] * np.log1p(popular_movies['review_count'])
        
        top_movie_ids = popular_movies.sort_values(by='score', ascending=False).head(n_recommendations).index
        
        recommended_movies = movies_df[movies_df['movieId'].isin(top_movie_ids)]
        recommended_movies = recommended_movies.set_index('movieId').loc[top_movie_ids].reset_index()
        
        return recommended_movies[['movieId', 'title', 'genres']]

# Test with a fake/new user ID that doesn't exist in our matrix
new_user_recommendations = get_recommendations_with_cold_start(99999, model_knn, user_movie_matrix, filtered_ratings, movies_df, n_recommendations=5)
print("\n--- Cold Start Recommendations for New User ---")
print(new_user_recommendations)

User ID 99999 is a new user (Cold Start)! Falling back to popular recommendations.

--- Cold Start Recommendations for New User ---
   movieId                             title                       genres
0      318  Shawshank Redemption, The (1994)                  Crime|Drama
1      356               Forrest Gump (1994)     Comedy|Drama|Romance|War
2      296               Pulp Fiction (1994)  Comedy|Crime|Drama|Thriller
3     2571                Matrix, The (1999)       Action|Sci-Fi|Thriller
4      593  Silence of the Lambs, The (1991)        Crime|Horror|Thriller


C:\Users\HP\AppData\Local\Temp\ipykernel_1324\958825371.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  popular_movies['score'] = popular_movies['mean_rating'] * np.log1p(popular_movies['review_count'])



# 🎬 Movie Recommendation System

A machine learning movie recommendation engine built using **Collaborative Filtering (K-Nearest Neighbors)** and a **Popularity-Based Cold Start Fallback** on the MovieLens dataset.

## 📌 1. Business Context & Objective

Streaming platforms such as Netflix rely on personalized recommendations to improve user engagement, customer satisfaction, and retention.

The objective of this project is to build an end-to-end movie recommendation system that analyzes historical user ratings and recommends the **Top 5 movies** based on individual user preferences.

## 📊 2. Dataset Overview

This project uses the **MovieLens (ml-latest-small)** dataset provided by GroupLens Research.

The dataset consists of two main files:

- **ratings.csv:** Contains user interactions, including User IDs, Movie IDs, rating scores (0.5–5.0), and timestamps.
- **movies.csv:** Contains movie metadata, including Movie IDs, movie titles, and genres.

## 🛠️ 3. Project Pipeline & Methodology

### 3.1 Data Acquisition & Loading

- Extracted and loaded the MovieLens dataset into Pandas DataFrames.
- Inspected data types, missing values, and dataset structure.
- Verified the integrity of the dataset before analysis.

### 3.2 Data Sparsity Analysis & Cleaning

- Calculated the user-item matrix sparsity to understand the distribution of user ratings.
- Filtered users with at least **20 ratings**.
- Retained movies with at least **10 reviews**.
- Reduced unnecessary noise and improved the quality of the recommendation data.

### 3.3 User-Item Matrix & Sparse Representation

- Transformed the filtered ratings into a user-item matrix.
- Rows represent users, columns represent movies, and values represent ratings.
- Converted the matrix into a **Compressed Sparse Row (CSR)** format using `scipy.sparse` to optimize memory usage and computational efficiency.

### 3.4 Collaborative Filtering Model (KNN)

Implemented an item-user K-Nearest Neighbors model using:

- **Algorithm:** Brute Force
- **Similarity Metric:** Cosine Similarity
- **Number of Neighbors:** 20

The model identifies users with similar rating patterns and uses their preferences to generate personalized movie recommendations.

### 3.5 Recommendation Engine

- Identified movies that the target user had not previously rated.
- Extracted recommendations from neighboring users.
- Ranked candidate movies using a combined rating and popularity score.

The recommendation score is calculated as:

$$
Score = Mean\ Rating \times \log(1 + Review\ Count)
$$

This approach balances movie quality with community popularity, reducing the likelihood of recommending movies based on a small number of ratings.

The system returns the **Top 5 recommended movies**, including their titles and genres.

### 3.6 Cold-Start Problem Handling

Implemented a popularity-based fallback mechanism for new or unrecognized users who do not have sufficient rating history.

The fallback recommends highly rated and popular movies using average ratings and review counts.

This ensures that the system can still generate recommendations when personalized user information is unavailable.

## 📈 4. Model Evaluation

The recommendation model was evaluated using an **80/20 Train-Test Split** and Root Mean Squared Error (RMSE).

| Evaluation Metric | Result | Interpretation |
|---|---:|---|
| RMSE | **1.0738** | Predictions deviate from actual user ratings by approximately 1.07 stars on a 1–5 rating scale. |

The RMSE provides an indication of the prediction error, with lower values representing better predictive accuracy.

## 🚀 5. Installation & Usage

### Prerequisites

Install the required Python libraries:

```bash
pip install pandas numpy scikit-learn scipy
```

### Dataset Setup

1. Download the MovieLens `ml-latest-small` dataset.
2. Extract the dataset into your project directory.
3. Ensure that `ratings.csv` and `movies.csv` are accessible inside the `ml-latest-small` folder.

### Python Implementation Overview

```python
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# 1. Load Data
ratings_df = pd.read_csv('ml-latest-small/ratings.csv')
movies_df = pd.read_csv('ml-latest-small/movies.csv')

# 2. Filter Users
counts_user = ratings_df['userId'].value_counts()
active_users = counts_user[counts_user >= 20].index

filtered_ratings = ratings_df[
    ratings_df['userId'].isin(active_users)
]

# 3. Filter Movies
counts_movie = filtered_ratings['movieId'].value_counts()
popular_movies = counts_movie[counts_movie >= 10].index

filtered_ratings = filtered_ratings[
    filtered_ratings['movieId'].isin(popular_movies)
]

# 4. Create User-Item Matrix
user_movie_matrix = filtered_ratings.pivot(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

# 5. Convert to Sparse Matrix
user_movie_sparse = csr_matrix(user_movie_matrix.values)

# 6. Train KNN Model
model_knn = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=20,
    n_jobs=-1
)

model_knn.fit(user_movie_sparse)

# 7. Generate Recommendations
recommendations_user_2 = get_movie_recommendations(
    2,
    model_knn,
    user_movie_matrix,
    filtered_ratings,
    movies_df,
    n_recommendations=5
)

print(recommendations_user_2)
```

**Note:** The `get_movie_recommendations()` function represents the custom recommendation function implemented in the project notebook.

## 💡 6. Future Enhancements

Potential improvements to the recommendation system include:

- **Matrix Factorization (SVD):** Explore latent user preferences and movie characteristics to improve recommendation quality.
- **Content-Based Filtering:** Incorporate movie genres, tags, and descriptions to recommend movies with similar characteristics.
- **Hybrid Recommendation System:** Combine collaborative and content-based filtering to improve personalization.
- **Interactive Dashboard:** Develop a user-friendly interface using Streamlit or Flask.
- **Advanced Evaluation:** Incorporate Precision@K, Recall@K, and MAP@K to evaluate recommendation ranking performance.

## 🧰 7. Technologies Used

| Technology | Application |
|---|---|
| Python | Core programming language |
| Pandas | Data manipulation and preprocessing |
| NumPy | Numerical computations |
| SciPy | Sparse matrix representation |
| Scikit-learn | KNN model and machine learning |
| Jupyter Notebook | Development and experimentation |

## 👨‍💻 8. Project Outcome

This project demonstrates the application of machine learning techniques to a real-world recommendation problem.

It combines collaborative filtering, similarity-based modeling, popularity scoring, and cold-start handling to deliver personalized movie recommendations.

The project also demonstrates practical skills in data preprocessing, exploratory analysis, feature engineering, model development, and evaluation.
